# Диагностика настройки ключевых слов (English)

Этот ноутбук **не запускает tuning заново**. Он восстанавливает и визуализирует pipeline по сохранённым артефактам:

- evidence-кэш (`artifacts/keyword_tuning_notebooks/en/field-1/evidence/`)
- trace tuning (`scart_tuned_model.tuning.json`)

## Важно про «слова для каждого sample»

Алгоритм выбирает **один глобальный словарь** (`static-keywords`) для всех документов.

| Что показывает ноутбук | Смысл |
|---|---|
| **Кандидаты на train-документ** | слова, извлечённые attention/QA из *конкретного* документа |
| **Финальный словарь** | 4–8 слов, выбранных на **dev** для всего корпуса |
| **Coverage matrix** | в каких train-доках встречались финальные keywords |

Разделы ниже идут **в порядке pipeline**: pool → (optional enrich, **off by default**) → prescreen → SFFS → stability → finalize.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from tqdm.auto import tqdm

# --- paths / profile ---
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "untie").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LANGUAGE = "en"
SEED = 42
TUNING_RUN = "full_large_v3"  # full_v2 | full | full_large_v3

ARTIFACTS = PROJECT_ROOT / "experiments/analysis_results/keyword_tuning_task/en" / TUNING_RUN
TRACE_PATH = ARTIFACTS / "scart_tuned_model.tuning.json"
MODEL_PATH = ARTIFACTS / "scart_tuned_model.json"
EVIDENCE_DIR = PROJECT_ROOT / "artifacts/keyword_tuning_notebooks/en/field-1/evidence"
METRIC_CACHE = PROJECT_ROOT / "artifacts/keyword_tuning_notebooks/en/field-1/extraction_metrics.json"
OUTPUT_DIR = ARTIFACTS / "diagnostics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Параметры, с которыми был запущен full_large_v3 (должны совпадать с trace)
TUNING_PROFILE = {
    "min_document_support": 2,
    "max_candidates": 150,
    "enrich_train_references": False,
    "min_enriched_support": 6,
    "screen_top_k": 60,
    "min_activation_rate": 0.15,
    "harm_cap": 0.12,
    "stability_threshold": 0.2,
    "selection_policy": "union",
    "min_keywords": 4,
}

display(pd.Series({
    "project_root": str(PROJECT_ROOT),
    "trace": str(TRACE_PATH),
    "evidence_dir": str(EVIDENCE_DIR),
    "output_dir": str(OUTPUT_DIR),
}, name="config"))


In [ ]:
from untie.keyword_diagnostics import (
    audit_enrichment,
    audit_pool_aggregation,
    audit_prescreen,
    funnel_summary,
    keyword_train_coverage_rows,
    load_evidence_directory,
    parse_tuning_trace,
    per_document_candidate_rows,
    sffs_trace_rows,
    stability_selection_rows,
    strategy_from_trace_payload,
    build_weighted_keyword_map,
)
from untie.keyword_evidence import ExtractionMetricCache
from untie.keyword_training import (
    CachedKeywordSubsetEvaluator,
    MetricWeights,
    candidate_evidence_from_documents,
)
from untie.keyword_tuning import (
    ObjectiveConfig,
    aggregate_candidate_pool,
    build_inverted_index,
    deterministic_document_split,
    enrich_pool_from_train_references,
)

if not TRACE_PATH.exists():
    raise FileNotFoundError(f"Не найден trace: {TRACE_PATH}. Сначала выполните tuning run.")

trace = parse_tuning_trace(TRACE_PATH)
model = json.loads(MODEL_PATH.read_text(encoding="utf-8"))
selected_keywords = trace["keywords"]
selected_strategy = strategy_from_trace_payload(trace)

by_id = load_evidence_directory(EVIDENCE_DIR)
all_doc_ids = sorted(by_id)
split = deterministic_document_split(all_doc_ids, seed=SEED)
train_ids, dev_ids, test_ids = split["train"], split["dev"], split["test"]

train_evidence = candidate_evidence_from_documents(by_id[d] for d in train_ids)
pool = aggregate_candidate_pool(
    train_evidence,
    train_ids,
    min_document_support=TUNING_PROFILE["min_document_support"],
)
if TUNING_PROFILE.get("enrich_train_references", False):
    pool = enrich_pool_from_train_references(
        pool,
        by_id,
        train_ids,
        min_document_support=TUNING_PROFILE["min_enriched_support"],
    )
pool = pool[: TUNING_PROFILE["max_candidates"]]
pool_terms = [item.term for item in pool]

print(f"Train/dev/test: {len(train_ids)}/{len(dev_ids)}/{len(test_ids)}")
print(f"Selected strategy: {selected_strategy.name}")
print(f"Selected keywords ({len(selected_keywords)}): {selected_keywords}")


## 1. Кандидаты на каждом train-документе

Строка = одно слово, извлечённое attention/QA из документа. Это **ещё не** финальный словарь.


In [ ]:
doc_candidates = pd.DataFrame(per_document_candidate_rows(by_id, train_ids))
summary_by_doc = (
    doc_candidates.groupby("doc_id")
    .agg(
        candidate_count=("word", "size"),
        mean_attention=("attention_weight", "mean"),
        mean_score_diff=("score_difference", "mean"),
        references=("references", "first"),
    )
    .sort_values("candidate_count", ascending=False)
)

print(f"Train docs with candidates: {summary_by_doc.shape[0]} / {len(train_ids)}")
display(summary_by_doc.head(15))
display(doc_candidates.sort_values(["doc_id", "attention_weight"], ascending=[True, False]).head(20))

# Топ слов по числу train-документов, где они были извлечены как кандидаты
term_doc_freq = (
    doc_candidates.assign(term=doc_candidates["word"].str.lower().str.strip())
    .groupby("term")["doc_id"].nunique()
    .sort_values(ascending=False)
    .head(25)
)
display(term_doc_freq.to_frame("train_doc_count"))
doc_candidates.to_csv(OUTPUT_DIR / "01_train_document_candidates.csv", index=False)
summary_by_doc.to_csv(OUTPUT_DIR / "01_train_document_candidate_summary.csv")


## 2. Агрегация candidate pool (train-only)

Показывает, какие термины **прошли** в pool и почему остальные отсеялись (`stopword`, `low_document_support`, `no_chunk_support`).


In [ ]:
pool_audit = pd.DataFrame(
    audit_pool_aggregation(
        train_evidence,
        train_ids,
        min_document_support=TUNING_PROFILE["min_document_support"],
    )
)
reason_counts = pool_audit.groupby(["decision", "reason"]).size().reset_index(name="count")
display(reason_counts)
display(pool_audit[pool_audit["decision"] == "kept"].head(20))

rejected_examples = pool_audit[pool_audit["decision"] == "rejected"].sort_values("document_support", ascending=False).head(20)
display(rejected_examples[["term", "reason", "document_support", "chunk_support_rate"]])

pool_audit.to_csv(OUTPUT_DIR / "02_pool_aggregation_audit.csv", index=False)


## 3. Обогащение pool из train references

Добавляет n-grams из gold task labels (train-only). Enriched terms часто имеют `chunk_support_rate=0`.


In [ ]:
enrich_audit = pd.DataFrame()
if TUNING_PROFILE.get("enrich_train_references", False):
    enrich_audit = pd.DataFrame(
        audit_enrichment(
            aggregate_candidate_pool(
                train_evidence,
                train_ids,
                min_document_support=TUNING_PROFILE["min_document_support"],
            ),
            by_id,
            train_ids,
            min_document_support=TUNING_PROFILE["min_enriched_support"],
        )
    )
    display(enrich_audit.groupby(["decision", "reason"]).size().reset_index(name="count"))
    display(enrich_audit[enrich_audit["decision"] == "kept"].head(20))
    display(enrich_audit[enrich_audit["decision"] == "rejected"].head(15))
    enrich_audit.to_csv(OUTPUT_DIR / "03_enrichment_audit.csv", index=False)
else:
    display(Markdown("Enrichment **отключён** — pool только из текстовых кандидатов (attention/QA)."))


## 4. Prescreen (single-keyword eval на dev)

Каждый терм из pool оценивается **в одиночку** на dev. Отсев по activation / harm / gain / top-k.

> Эта ячейка может занять несколько минут: ~150 полных прогонов evaluator.


In [ ]:
keyword_map = build_weighted_keyword_map(pool, by_id, train_ids)

metric_cache = ExtractionMetricCache(METRIC_CACHE)
evaluator = CachedKeywordSubsetEvaluator(
    by_id,
    keyword_map,
    metric_cache,
    language=LANGUAGE,
    strategy=selected_strategy,
    include_bertscore=False,
    metric_weights=MetricWeights.exact_match(),
)
objective_config = ObjectiveConfig(
    downside_penalty=0.75,
    harm_penalty=0.5,
    fallback_penalty=0.1,
    size_penalty=0.002,
    harm_threshold=0.01,
    confidence_weight=0.25,
    bootstrap_seed=SEED,
    activation_weight=0.12,
    min_activation_rate=TUNING_PROFILE["min_activation_rate"],
    use_conditional_gain=True,
    inactive_fallback_penalty=0.04,
    win_rate_weight=0.08,
    apply_activation_gate=True,
)
inverted_index = build_inverted_index(pool)

prescreen_audit = pd.DataFrame(
    audit_prescreen(
        pool_terms,
        evaluator,
        dev_ids,
        objective_config,
        top_k=TUNING_PROFILE["screen_top_k"],
        harm_cap=TUNING_PROFILE["harm_cap"],
        inverted_index=inverted_index,
    )
)
display(prescreen_audit.groupby(["decision", "reason"]).size().reset_index(name="count"))
kept = prescreen_audit[prescreen_audit["decision"] == "kept"].sort_values("rank")
display(kept.head(25))
rejected = prescreen_audit[prescreen_audit["decision"] == "rejected"].sort_values("mean_gain_active", ascending=False).head(25)
display(rejected[["term", "reason", "mean_gain_active", "activation_rate", "harm_rate", "rank"]])
prescreen_audit.to_csv(OUTPUT_DIR / "04_prescreen_audit.csv", index=False)


## 5. SFFS (пошаговый trace выбранной стратегии)

Каждая stability run добавляет/удаляет keywords на dev-panel. Ниже — все шаги для **выбранной** стратегии.


In [ ]:
sffs_df = pd.DataFrame(sffs_trace_rows(trace))
if sffs_df.empty:
    raise RuntimeError("SFFS trace пуст — проверьте trace JSON")

finals = sffs_df.sort_values(["run", "evaluations_used"]).groupby("run").tail(1)
display(finals[["run", "subset", "objective", "evaluations_used"]])

fig, ax = plt.subplots(figsize=(10, 4))
for run, group in sffs_df.groupby("run"):
    ax.plot(group["evaluations_used"], group["objective"], marker="o", label=f"run {run}")
ax.set_title(f"SFFS objective — {selected_strategy.name}")
ax.set_xlabel("evaluations used")
ax.set_ylabel("objective")
ax.legend()
plt.tight_layout()
plt.show()

sffs_df.to_csv(OUTPUT_DIR / "05_sffs_trace.csv", index=False)


## 6. Stability selection и finalize

- **Stability runs** — финальные subset каждого run
- **Threshold** — термин остаётся, если выбран в ≥ `stability_threshold` доле runs
- **Finalize** — union/relaxed/prune; итог в `trace['keywords']`


In [ ]:
stability_df = pd.DataFrame(
    stability_selection_rows(trace, stability_threshold=TUNING_PROFILE["stability_threshold"])
)
run_rows = stability_df[stability_df["stage"] == "stability_run"]
threshold_rows = stability_df[stability_df["stage"] == "stability_threshold"]
display(run_rows)
display(threshold_rows)

meta = pd.DataFrame(trace.get("keyword_metadata", []))
display(meta)

print("Finalize policy:", TUNING_PROFILE["selection_policy"])
print("Final keywords:", selected_keywords)

stability_df.to_csv(OUTPUT_DIR / "06_stability_audit.csv", index=False)
meta.to_csv(OUTPUT_DIR / "06_selected_keyword_metadata.csv", index=False)


## 7. Покрытие финальных keywords по train-документам

Heatmap: строка = keyword, столбец = train doc (1 = терм встречался в evidence этого документа).


In [ ]:
coverage = pd.DataFrame(keyword_train_coverage_rows(pool, selected_keywords, train_ids))
matrix = coverage.pivot(index="keyword", columns="doc_id", values="present_in_train_evidence").fillna(False)

# Для читаемости показываем только docs, где есть хотя бы один selected keyword
active_docs = matrix.columns[matrix.any(axis=0)]
matrix_active = matrix[active_docs]
print(f"Train docs with any selected keyword in evidence: {matrix_active.shape[1]} / {len(train_ids)}")

fig, ax = plt.subplots(figsize=(min(24, 0.12 * matrix_active.shape[1] + 4), 4))
sns.heatmap(matrix_active.astype(int), cmap="Blues", cbar=False, ax=ax)
ax.set_title("Selected keywords × train documents (evidence presence)")
plt.tight_layout()
plt.show()

coverage.to_csv(OUTPUT_DIR / "07_keyword_train_coverage.csv", index=False)

funnel_payload = {
    "pool": pool_audit.to_dict("records"),
    "prescreen": prescreen_audit.to_dict("records"),
    "stability_terms": threshold_rows.to_dict("records"),
}
if not enrich_audit.empty:
    funnel_payload["enrich"] = enrich_audit.to_dict("records")
funnel = pd.DataFrame(funnel_summary(funnel_payload))
display(funnel)
funnel.to_csv(OUTPUT_DIR / "00_pipeline_funnel.csv", index=False)

print(f"CSV exports written to {OUTPUT_DIR}")
